## 0.Rdkit验证SMILES手性

In [ ]:
'''
功能:rdkit验证手性立体化学
ligand1:https://www.ebi.ac.uk/chebi/CHEBI:62456
老师给的(手性有问题): C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/Cucurbitadienol#section=3D-Conformer

ligand2(P450环氧):https://www.ebi.ac.uk/chebi/CHEBI:229949
老师给的(手性有问题): C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/171037431#section=3D-Conformer

ligand1 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
'''
from rdkit import Chem
from rdkit.Chem import AllChem

ligand1_teacher = "C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C"
ligand1_chebi = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2_teacher = "C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C"
ligand2_chebi   = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
ligand3_chebi = "C[C@H](CC[C@H](C(C)(C)O)O)[C@H]1CC[C@@]2([C@@]1(CC[C@@]3([C@H]2CC=C4[C@H]3CC[C@@H](C4(C)C)O)C)C)C"

for name, smi in [("teacher", ligand2_teacher), ("chebi", ligand3_chebi)]:
    mol = Chem.MolFromSmiles(smi)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    print(name)
    print("分子式:", Chem.rdMolDescriptors.CalcMolFormula(mol))
    print("规范SMILES:", Chem.MolToSmiles(mol))
    print("手性中心 (原子序号, R/S 或 ?未指定):", centers)
    print()

teacher
分子式: C30H50O2
规范SMILES: C[C@H](CCC1OC1(C)C)C1CC[C@@]2(C)C3CC=C4C(CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, '?'), (9, '?'), (12, 'S'), (14, '?'), (21, 'S'), (25, '?'), (26, 'R'), (30, 'R')]

chebi
分子式: C30H52O3
规范SMILES: C[C@H](CC[C@@H](O)C(C)(C)O)[C@H]1CC[C@@]2(C)[C@@H]3CC=C4[C@@H](CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, 'R'), (10, 'R'), (13, 'S'), (14, 'R'), (17, 'R'), (18, 'R'), (22, 'S'), (25, 'S')]



## 1. 如果没找到官方的sdf就用这个

In [ ]:
'''
功能：SMILES转sdf
'''
from rdkit import Chem
from rdkit.Chem import AllChem
smi_chebi  = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
for smiles, name in [(ligand2_chebi, "ligand1")]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"{name}: SMILES解析失败，跳过")
        break

    # 检查手性完整性，未指定的中心先显示？
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    unassigned = [c for c in centers if c[1] == '?']
    if unassigned:
        print(f"{name}存在未指定手性中心: {unassigned}，构象将由算法任意指定，需核实SMILES")

    mol.SetProp("_Name", name)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
    if len(cids) == 0:
        print(f"{name}: 标准ETKDGv3生成失败，尝试随机坐标初始化")
        params.useRandomCoords = True
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
        if len(cids) == 0:
            print(f"{name}: 3D构象生成彻底失败，跳过该分子")
            break

    # 力场优化，选能量最低的构象
    energies = []
    for cid in cids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(
                mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid
            )
            ff.Minimize(maxIts=2000)
            energies.append((cid, ff.CalcEnergy()))
        except Exception as e:
            print(f"{name}: 构象{cid}优化报错: {e}")

    if not energies:
        print(f"{name}: 所有构象优化失败，跳过")
        break

    best_cid = min(energies, key=lambda x: x[1])[0]
    print(f"{name}: {len(centers)}个手性中心--->{centers}，最优构象能量 = {min(e for _, e in energies):.2f} kcal/mol")

    writer = Chem.SDWriter(f"{name}_input.sdf")
    writer.write(mol, confId=best_cid)
    writer.close()
    print(f"已生成: {name}_input.sdf\n")

ligand1: 9个手性中心 → [(0, 'R'), (7, 'S'), (11, 'S'), (12, 'R'), (16, 'R'), (18, 'S'), (22, 'R'), (23, 'R'), (27, 'S')]，最优构象能量 = 146.39 kcal/mol
已生成: ligand1_input.sdf



## 2. 突变点位确定

In [ ]:
# ligand2_5A
list1 = "PHE33,PRO34,ASP101,TRP102,ILE105,VAL126,TYR150,MET151,PHE154,LEU172,THR175,SER176,ASN179,ALA183,PRO185,LEU187,PHE193,ILE196,TYR230,LEU261,THR262,HIS295,PHE296".split(',')
# ligand3_5A
list2 = "PHE33,PRO34,ASP101,TRP102,ILE105,SER125,VAL126,TYR150,MET151,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,TYR230,LEU261,THR262,PHE265,HIS295,PHE296".split(',')
# caver
list3 = "VAL126,ILE105,HIS100,ALA104,SER125,THR262".split(',')

# 关键motif和催化残基
critical_residues = "ASP101,HIS295,ASP260,TYR150,TYR230,HIS31,GLY32,PHE33,PRO34,TRP102".split(',')

# 将列表转换为集合
set1 = set(list1)
set2 = set(list2)
set3 = set(list3)
set_critical = set(critical_residues)

# 求前三项的并集
union_set = set1 | set2 | set3

# 剔除关键残基求差集(A中去掉B)
final_set = union_set - set_critical

# 按氨基酸后面的数字序号进行排序
final_sorted = sorted(list(final_set), key=lambda x: int(x[3:]))

print("最终需要突变的残基列表:")
print(",".join(final_sorted))
print(f"共计: {len(final_sorted)}个残基")

最终需要突变的残基列表:
HIS100,ALA104,ILE105,SER125,VAL126,MET151,PHE154,LEU172,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,LEU261,THR262,PHE265,PHE296

共计: 22个残基


## 3. 生成饱和突变序列列表共foldx使用

In [ ]:
'''
功能:对残基名字对应改成单个字母
'''
import re

# 把PyMOL打印出来的那一串直接粘贴在引号里
pymol_output = "HIS100,ALA104,ILE105,SER125,VAL126,MET151,PHE154,LEU172,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,LEU261,THR262,PHE265,PHE296"
# 氨基酸3字母转1字母字典
aa_map = {
    'ALA':'A', 'CYS':'C', 'ASP':'D', 'GLU':'E', 'PHE':'F', 'GLY':'G', 'HIS':'H', 
    'ILE':'I', 'LYS':'K', 'LEU':'L', 'MET':'M', 'ASN':'N', 'PRO':'P', 'GLN':'Q', 
    'ARG':'R', 'SER':'S', 'THR':'T', 'VAL':'V', 'TRP':'W', 'TYR':'Y'
}

# 清理逗号，分割字符串
raw_list = re.split(r'[,\s]+', pymol_output.strip())
# 去除空项
raw_list = [x for x in raw_list if x]
print(f"要突变的位点总数{len(raw_list)}")

residues = []
aas = []
for item in raw_list:
    # 提取字母部分PHE
    aa_3 = "".join(filter(str.isalpha, item)).upper()
    # 提取数字部分105
    res_num = "".join(filter(str.isdigit, item))
    if aa_3 in aa_map and res_num:
        residues.append(int(res_num))
        aas.append(aa_map[aa_3])
    else:
        print(f"无法识别的项 '{item}'，已跳过")

# 输出可以直接复制的结果
print(f"residues_to_mutate = {residues}")
print(f"original_aas = {aas}")

'''
2. 功能:对所有残基进行单点饱和突变,并写成标准格式到individual_list.txt文件中
'''
# 填入22个残基的编号
residues_to_mutate = residues

# 填入这22个残基原本的氨基酸单字母必须和编号一一对应
# 举例，F对应105，L对应108
original_aas = aas

# 这里的链ID通常是A，如果蛋白是B链，就改成'B'
chain_id = 'A'

# 目标：20种标准氨基酸
all_aas = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 
           'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

with open("individual_list.txt", "w") as f:
    for i in range(len(residues_to_mutate)):
        res_num = residues_to_mutate[i]
        wt_aa = original_aas[i]
        
        for mutant_aa in all_aas:
            # 跳过突变成自己的
            if mutant_aa == wt_aa: continue 
            # FoldX格式：原AA + 链 + 编号 + 新AA + 分号
            # F=原氨基酸单字母，A=链ID，105=编号，A=突变后氨基酸，分号结尾
            line = f"{wt_aa}{chain_id}{res_num}{mutant_aa};\n"
            f.write(line)
total_mutations = len(residues_to_mutate) * 19
print(f"共生成{total_mutations}个单点突变任务")
print("已生成individual_list.txt")

要突变的位点总数22
residues_to_mutate = [100, 104, 105, 125, 126, 151, 154, 172, 175, 176, 179, 181, 183, 185, 186, 187, 193, 196, 261, 262, 265, 296]
original_aas = ['H', 'A', 'I', 'S', 'V', 'M', 'F', 'L', 'T', 'S', 'N', 'S', 'A', 'P', 'C', 'L', 'F', 'I', 'L', 'T', 'F', 'F']
共生成418个单点突变任务
已生成individual_list.txt


## 4.分析vina结果筛选